# 하이브리드 앙상블 검색 — weighted RRF

BM25의 어휘 검색과 임베딩의 의미 검색을 결합합니다. v1 핵심 패키지에는 예전
`EnsembleRetriever`가 없으므로, 안정적인 문서 ID를 사용해 weighted Reciprocal
Rank Fusion(RRF)을 직접 구현하고 Runnable로 노출합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-core==1.6.3" "langchain-openai==1.6.2" \
#   "langchain-chroma==1.1.0" python-dotenv rank-bm25 numpy


In [ ]:
import getpass
import os
import re
from collections import defaultdict
from uuid import uuid4

import numpy as np
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from rank_bm25 import BM25Okapi

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
texts = [
    "I like apples",
    "I like Apple the company",
    "I like Apple's iPhone",
    "Apple is my favorite company",
    "I like Apple's iPad",
    "I like Apple's MacBook",
]
docs = [
    Document(page_content=text, metadata={"doc_id": f"doc-{index}"})
    for index, text in enumerate(texts)
]


def tokenize_english(text: str) -> list[str]:
    return re.findall(r"[a-z0-9']+", text.lower())


bm25_index = BM25Okapi([tokenize_english(doc.page_content) for doc in docs])


def sparse_search(query: str, k: int = 4) -> list[Document]:
    scores = bm25_index.get_scores(tokenize_english(query))
    order = np.argsort(scores)[::-1][:k]
    return [docs[index] for index in order]


sparse_retriever = RunnableLambda(sparse_search)
dense_store = Chroma.from_documents(
    docs,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"ensemble-demo-{uuid4().hex}",
    ids=[doc.metadata["doc_id"] for doc in docs],
    collection_configuration=CHROMA_CONFIGURATION,
)
dense_retriever = dense_store.as_retriever(search_kwargs={"k": 4})


## weighted RRF

RRF는 서로 다른 점수 척도를 직접 더하지 않고 순위만 결합합니다.
각 검색기의 기여는 `weight / (c + rank)`이며, `c=60`은 흔히 쓰는 완화 상수입니다.


In [ ]:
def weighted_rrf(
    query: str,
    retrievers: list,
    weights: list[float],
    *,
    k: int = 4,
    c: int = 60,
) -> list[Document]:
    if len(retrievers) != len(weights):
        raise ValueError("retrievers와 weights의 길이가 같아야 합니다.")
    if not np.isclose(sum(weights), 1.0):
        raise ValueError("weights의 합은 1이어야 합니다.")

    scores: defaultdict[str, float] = defaultdict(float)
    documents: dict[str, Document] = {}
    for retriever, weight in zip(retrievers, weights):
        for rank, doc in enumerate(retriever.invoke(query), start=1):
            doc_id = doc.metadata["doc_id"]
            documents[doc_id] = doc
            scores[doc_id] += weight / (c + rank)

    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return [
        Document(
            page_content=documents[doc_id].page_content,
            metadata={
                **documents[doc_id].metadata,
                "rrf_score": scores[doc_id],
            },
        )
        for doc_id in ranked_ids
    ]


def ensemble_search(request: dict) -> list[Document]:
    return weighted_rrf(
        request["query"],
        [sparse_retriever, dense_retriever],
        request.get("weights", [0.7, 0.3]),
        k=request.get("k", 4),
    )


ensemble_retriever = RunnableLambda(ensemble_search).with_config(
    {"run_name": "weighted_rrf"}
)


In [ ]:
def show(label: str, results: list[Document]) -> None:
    print(f"\n[{label}]")
    for doc in results:
        score = doc.metadata.get("rrf_score")
        suffix = f" (RRF={score:.6f})" if score is not None else ""
        print(f"- {doc.page_content}{suffix}")


query = "my favorite fruit is apple"
show("Sparse / BM25", sparse_retriever.invoke(query))
show("Dense / Chroma", dense_retriever.invoke(query))
show(
    "Hybrid / 0.7 sparse + 0.3 dense",
    ensemble_retriever.invoke({"query": query, "weights": [0.7, 0.3]}),
)


## 런타임 가중치 변경

가중치를 숨은 config 필드가 아니라 요청 스키마에 넣어 호출자가 무엇을
바꾸는지 명확하게 만듭니다.


In [ ]:
sparse_only = ensemble_retriever.invoke(
    {"query": query, "weights": [1.0, 0.0], "k": 3}
)
dense_only = ensemble_retriever.invoke(
    {"query": query, "weights": [0.0, 1.0], "k": 3}
)

show("Sparse only", sparse_only)
show("Dense only", dense_only)
